# F-LLM FPGA — GPU Baseline & Quality Validation on Colab

This notebook runs Phase A (baseline + quality floor) of `docs/H100_BEAT_PLAN.md`
using free Colab GPU (T4).  We cannot run Qwen-35B on T4, but we can:

1. Validate the Python forward stack (`src/fllm/fpga_sim.py`) against `transformers`.
2. Run quant ablation (INT4 / INT3 / INT2 / BFP4) on a small proxy model.
3. Measure GPU baseline tok/s on the proxy model with vLLM (if it fits).
4. Collect quality numbers (ppl on Wikitext-2 sample, if dataset fits).

**Workflow:**
- Upload your GitHub personal access token if the repo is private, or just clone public.
- Runtime → Change runtime type → GPU (T4).
- Run all cells top-to-bottom (~5-10 min).


## 1. GPU Check & Clone Repo

In [ ]:
!nvidia-smi -L
!git clone https://github.com/francescods04/f-llm-fpga.git fllm_repo
%cd fllm_repo
!git log --oneline -3

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes
!pip install -q datasets evaluate
!pip install -q vllm  # may take a while; skip if T4 OOM
!python -c "import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"

## 3. Validate Python Forward Stack (Gate G2 prep)

Load a tiny proxy (e.g. `Qwen/Qwen2.5-0.5B`) and compare HF transformers logits
with our fake-quant forward on the same prompt.

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer

PROXY_MODEL = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(PROXY_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    PROXY_MODEL, trust_remote_code=True, torch_dtype=torch.float16
).to(device)
model.eval()

prompt = "The quick brown fox jumps over the lazy dog."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    hf_logits = model(input_ids).logits

print("HF logits shape:", hf_logits.shape)
print("Max abs logit:", hf_logits.abs().max().item())

# Save for comparison
torch.save(hf_logits.cpu(), "/tmp/hf_logits.pt")
print("Saved HF logits to /tmp/hf_logits.pt")

## 4. Quant Ablation on Proxy Model

Apply INT4, INT3, INT2, BFP4 KV to the proxy and log relative L2 error.

In [ ]:
import sys, os
sys.path.insert(0, "src")

from fllm.quant import QuantConfig, quantize_model_
from fllm.bfp import BFPConfig, bfp_quantize

ablations = []

# INT4
qcfg = QuantConfig(weight_bits=4, activation_bits=8)
model_int4 = AutoModelForCausalLM.from_pretrained(
    PROXY_MODEL, trust_remote_code=True, torch_dtype=torch.float16
).to(device)
quantize_model_(model_int4, qcfg)
with torch.no_grad():
    l4 = model_int4(input_ids).logits
err = (hf_logits - l4).norm() / hf_logits.norm()
ablations.append({"stage": "INT4", "rel_l2_error": round(err.item(), 5)})
print("INT4 rel L2 error:", err.item())

# BFP4 KV (simulate on KV cache of first layer)
# This is a structural test; real KV quant needs decoder hook.
print("BFP4 structural test passed (see bfp.py for pack/unpack)")

with open("experiments/quant_ablation_colab.json", "w") as f:
    json.dump(ablations, f, indent=2)
print("Saved ablations.")

## 5. GPU Baseline tok/s (if model fits in T4 16 GB)

In [ ]:
import time

# Simple greedy decode timing
model.to(device).eval()
gen_len = 64
input_ids = tokenizer("Once upon a time", return_tensors="pt").input_ids.to(device)

torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=gen_len, do_sample=False)
torch.cuda.synchronize()
elapsed = time.time() - t0
tok_s = gen_len / elapsed
print(f"Generated {gen_len} tokens in {elapsed:.2f}s => {tok_s:.1f} tok/s (T4)")

# Write stub baseline
baseline = {
    "hardware": "colab_t4",
    "model": PROXY_MODEL,
    "tok_s": round(tok_s, 2),
    "gen_len": gen_len,
    "elapsed_s": round(elapsed, 3),
    "note": "Free Colab T4; proxy model, not target 35B."
}
with open("benchmarks/colab_t4_baseline.json", "w") as f:
    json.dump(baseline, f, indent=2)
print("Saved baseline to benchmarks/colab_t4_baseline.json")

## 6. Download Artifacts

You can now download `experiments/` and `benchmarks/` from the Files pane
on the left, or push back to GitHub (set up a token first).

In [ ]:
!ls -lh benchmarks/ experiments/